# Car Insurance Premium Prediction ML Pipeline

This notebook implements an end-to-end ML pipeline for predicting car insurance premiums using:
- **Snowflake Feature Store** (with Online Feature Store for real-time serving)
- **XGBoost** model with StandardScaler preprocessing
- **Snowflake Model Registry** for model management
- **SPCS Inference Service** for real-time predictions

In [ ]:
!pip install snowflake-ml-python==1.29.0 --quiet
!pip install snowflake-snowpark-python==1.47.0 --quiet
!pip install snowflake-connector-python==3.18.0 --quiet


In [ ]:
!pip list | grep snowflake

## 1. Setup and Configuration

In [1]:
import os
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
import random

from snowflake.snowpark import Session
from snowflake.snowpark import functions as F
from snowflake.snowpark.types import StructType, StructField, StringType, IntegerType, FloatType, DateType

try:
    from snowflake.snowpark.context import get_active_session
    session = get_active_session()
except Exception:
    session = Session.builder.config(
        "connection_name",
        os.getenv("SNOWFLAKE_CONNECTION_NAME") or "keypair"
    ).create()

In [2]:
DATABASE = "CC_ML_INSURANCE"
SCHEMA = "CAR_PRICING"

session.sql(f'CREATE OR REPLACE DATABASE {DATABASE}').collect()
session.sql(f'USE DATABASE {DATABASE}').collect()
session.sql(f'CREATE OR REPLACE SCHEMA {SCHEMA}').collect()
session.sql(f'USE SCHEMA {SCHEMA}').collect()


session.use_database(DATABASE)
session.use_schema(SCHEMA)

print(f"Connected to: {session.get_current_account()}")
print(f"Using: {DATABASE}.{SCHEMA}")

## 2. Generate Synthetic Car Insurance Data

In [3]:
np.random.seed(42)
random.seed(42)

N_CUSTOMERS = 5000
N_POLICIES = 8000

CAR_MAKES = {
    'Toyota': ['Camry', 'Corolla', 'RAV4', 'Highlander', 'Prius'],
    'Honda': ['Civic', 'Accord', 'CR-V', 'Pilot', 'Odyssey'],
    'Ford': ['F-150', 'Mustang', 'Explorer', 'Escape', 'Bronco'],
    'BMW': ['3 Series', '5 Series', 'X3', 'X5', 'M3'],
    'Mercedes': ['C-Class', 'E-Class', 'GLC', 'GLE', 'S-Class'],
    'Chevrolet': ['Silverado', 'Malibu', 'Equinox', 'Tahoe', 'Corvette'],
    'Tesla': ['Model 3', 'Model Y', 'Model S', 'Model X'],
    'Nissan': ['Altima', 'Rogue', 'Sentra', 'Pathfinder', 'Maxima']
}

COLORS = ['Black', 'White', 'Silver', 'Gray', 'Blue', 'Red', 'Green', 'Brown']
FUEL_TYPES = ['Gasoline', 'Diesel', 'Hybrid', 'Electric']
TRANSMISSIONS = ['Automatic', 'Manual', 'CVT']
COVERAGE_TYPES = ['Basic', 'Standard', 'Premium', 'Comprehensive']

BASE_PRICES = {
    'Toyota': 28000, 'Honda': 27000, 'Ford': 35000, 'BMW': 55000,
    'Mercedes': 60000, 'Chevrolet': 32000, 'Tesla': 50000, 'Nissan': 26000
}

print("Data constants defined")

In [4]:
customer_ids = [f"CUST_{str(i).zfill(6)}" for i in range(1, N_CUSTOMERS + 1)]
first_names = ['James', 'Mary', 'John', 'Patricia', 'Robert', 'Jennifer', 'Michael', 'Linda', 
               'William', 'Elizabeth', 'David', 'Susan', 'Richard', 'Jessica', 'Joseph', 'Sarah']
last_names = ['Smith', 'Johnson', 'Williams', 'Brown', 'Jones', 'Garcia', 'Miller', 'Davis',
              'Rodriguez', 'Martinez', 'Wilson', 'Anderson', 'Taylor', 'Thomas', 'Moore', 'Jackson']

customers_data = []
for cust_id in customer_ids:
    age = int(np.random.normal(42, 15))
    age = max(18, min(80, age))
    years_licensed = min(age - 16, int(np.random.exponential(15)))
    years_licensed = max(1, years_licensed)
    claims_history = int(np.random.exponential(0.8))
    claims_history = min(claims_history, 10)
    credit_score = int(np.random.normal(700, 80))
    credit_score = max(300, min(850, credit_score))
    
    customers_data.append({
        'CUSTOMER_ID': cust_id,
        'FIRST_NAME': random.choice(first_names),
        'LAST_NAME': random.choice(last_names),
        'AGE': age,
        'GENDER': random.choice(['M', 'F']),
        'YEARS_LICENSED': years_licensed,
        'CLAIMS_HISTORY': claims_history,
        'CREDIT_SCORE': credit_score,
        'STATE': random.choice(['CA', 'TX', 'FL', 'NY', 'IL', 'PA', 'OH', 'GA', 'NC', 'MI'])
    })

customers_df = pd.DataFrame(customers_data)
print(f"Generated {len(customers_df)} customers")
customers_df.head()

In [5]:
current_year = datetime.now().year

policies_data = []
for i in range(N_POLICIES):
    cust_id = random.choice(customer_ids)
    customer = customers_df[customers_df['CUSTOMER_ID'] == cust_id].iloc[0]
    
    car_make = random.choice(list(CAR_MAKES.keys()))
    car_model = random.choice(CAR_MAKES[car_make])
    car_year = random.randint(2010, current_year)
    car_age = current_year - car_year
    
    if car_make == 'Tesla':
        fuel_type = 'Electric'
    elif car_make in ['BMW', 'Mercedes']:
        fuel_type = random.choices(FUEL_TYPES, weights=[0.6, 0.15, 0.2, 0.05])[0]
    else:
        fuel_type = random.choices(FUEL_TYPES, weights=[0.7, 0.1, 0.15, 0.05])[0]
    
    transmission = random.choices(TRANSMISSIONS, weights=[0.7, 0.15, 0.15])[0]
    avg_km_per_year = np.random.normal(15000, 5000)
    kilometers = int(max(1000, car_age * avg_km_per_year + np.random.normal(0, 5000)))
    engine_size = random.choice([1.5, 1.8, 2.0, 2.4, 2.5, 3.0, 3.5, 4.0, 5.0])
    
    base_price = BASE_PRICES[car_make]
    depreciation = 0.85 ** car_age
    km_factor = max(0.5, 1 - (kilometers / 300000))
    car_value = base_price * depreciation * km_factor
    
    coverage_type = random.choice(COVERAGE_TYPES)
    deductible = random.choice([250, 500, 1000, 1500, 2000])
    
    base_premium = 800
    
    if customer['AGE'] < 25:
        age_factor = 1.5
    elif customer['AGE'] > 65:
        age_factor = 1.2
    else:
        age_factor = 1.0
    
    experience_factor = max(0.8, 1.3 - (customer['YEARS_LICENSED'] * 0.02))
    claims_factor = 1 + (customer['CLAIMS_HISTORY'] * 0.15)
    credit_factor = max(0.8, 1.3 - ((customer['CREDIT_SCORE'] - 600) / 500))
    
    car_value_factor = 0.03 * (car_value / 10000)
    make_factor = {'BMW': 1.3, 'Mercedes': 1.35, 'Tesla': 1.25, 'Chevrolet': 1.05,
                   'Ford': 1.1, 'Toyota': 0.95, 'Honda': 0.95, 'Nissan': 1.0}.get(car_make, 1.0)
    
    coverage_factor = {'Basic': 0.7, 'Standard': 1.0, 'Premium': 1.3, 'Comprehensive': 1.6}[coverage_type]
    deductible_factor = {250: 1.2, 500: 1.1, 1000: 1.0, 1500: 0.9, 2000: 0.85}[deductible]
    
    annual_premium = base_premium * age_factor * experience_factor * claims_factor * credit_factor
    annual_premium = annual_premium * (1 + car_value_factor) * make_factor
    annual_premium = annual_premium * coverage_factor * deductible_factor
    annual_premium = annual_premium + np.random.normal(0, 50)
    annual_premium = max(400, min(5000, annual_premium))
    
    policy_id = f"POL_{str(i+1).zfill(7)}"
    start_date = datetime.now() - timedelta(days=random.randint(0, 365))
    
    policies_data.append({
        'POLICY_ID': policy_id,
        'CUSTOMER_ID': cust_id,
        'CAR_MAKE': car_make,
        'CAR_MODEL': car_model,
        'CAR_YEAR': car_year,
        'COLOR': random.choice(COLORS),
        'KILOMETERS': kilometers,
        'ENGINE_SIZE': engine_size,
        'FUEL_TYPE': fuel_type,
        'TRANSMISSION': transmission,
        'COVERAGE_TYPE': coverage_type,
        'DEDUCTIBLE': deductible,
        'ESTIMATED_CAR_VALUE': round(car_value, 2),
        'ANNUAL_PREMIUM': round(annual_premium, 2),
        'POLICY_START_DATE': start_date.strftime('%Y-%m-%d'),
        'UPDATED_AT': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    })

policies_df = pd.DataFrame(policies_data)
print(f"Generated {len(policies_df)} policies")
print(f"Premium range: ${policies_df['ANNUAL_PREMIUM'].min():.2f} - ${policies_df['ANNUAL_PREMIUM'].max():.2f}")
print(f"Mean premium: ${policies_df['ANNUAL_PREMIUM'].mean():.2f}")
policies_df.head()

In [6]:
session.write_pandas(customers_df, "CUSTOMERS", auto_create_table=True, overwrite=True)
session.write_pandas(policies_df, "POLICIES", auto_create_table=True, overwrite=True)

print("Data uploaded to Snowflake:")
print(f"  - {DATABASE}.{SCHEMA}.CUSTOMERS: {session.table('CUSTOMERS').count()} rows")
print(f"  - {DATABASE}.{SCHEMA}.POLICIES: {session.table('POLICIES').count()} rows")

## 3. Create Feature Store with Online Features

In [7]:
from snowflake.ml.feature_store import FeatureStore, FeatureView, Entity, CreationMode
from snowflake.ml.feature_store.feature_view import OnlineConfig

fs = FeatureStore(
    session=session,
    database=DATABASE,
    name=SCHEMA,
    default_warehouse='COMPUTE_WH',
    creation_mode=CreationMode.CREATE_IF_NOT_EXIST
)

print(f"Feature Store created: {DATABASE}.{SCHEMA}")

In [8]:
customer_entity = Entity(
    name="CUSTOMER",
    join_keys=["CUSTOMER_ID"],
    desc="Customer entity for car insurance"
)

fs.register_entity(customer_entity)
print("Entity 'CUSTOMER' registered")

fs.list_entities().to_pandas()

In [9]:
current_year_val = datetime.now().year

customer_features_sql = f"""
SELECT 
    c.CUSTOMER_ID,
    c.AGE,
    c.YEARS_LICENSED,
    c.CLAIMS_HISTORY,
    c.CREDIT_SCORE,
    
    -- Risk Score: Higher for young/old drivers, many claims, low credit
    ROUND(
        (CASE WHEN c.AGE < 25 THEN 30 WHEN c.AGE > 65 THEN 20 ELSE 0 END) +
        (c.CLAIMS_HISTORY * 15) +
        (GREATEST(0, 40 - c.YEARS_LICENSED)) +
        (GREATEST(0, (700 - c.CREDIT_SCORE) / 10))
    , 2) AS RISK_SCORE,
    
    -- Average claims per year licensed
    ROUND(c.CLAIMS_HISTORY / GREATEST(1, c.YEARS_LICENSED), 4) AS AVG_CLAIMS_PER_YEAR,
    
    -- Credit tier
    CASE 
        WHEN c.CREDIT_SCORE >= 750 THEN 'Excellent'
        WHEN c.CREDIT_SCORE >= 700 THEN 'Good'
        WHEN c.CREDIT_SCORE >= 650 THEN 'Fair'
        ELSE 'Poor'
    END AS CREDIT_TIER,
    
    -- Total policies
    COUNT(p.POLICY_ID) AS TOTAL_POLICIES,
    
    -- Average car age across all policies
    ROUND(AVG({current_year_val} - p.CAR_YEAR), 2) AS AVG_CAR_AGE,
    
    -- Average kilometers across all cars
    ROUND(AVG(p.KILOMETERS), 0) AS AVG_KILOMETERS,
    
    -- Total estimated car value
    ROUND(SUM(p.ESTIMATED_CAR_VALUE), 2) AS TOTAL_CAR_VALUE,
    
    -- Average deductible chosen by customer
    ROUND(AVG(p.DEDUCTIBLE), 0) AS AVG_DEDUCTIBLE,
    
    CURRENT_TIMESTAMP() AS UPDATED_AT
    
FROM {DATABASE}.{SCHEMA}.CUSTOMERS c
LEFT JOIN {DATABASE}.{SCHEMA}.POLICIES p ON c.CUSTOMER_ID = p.CUSTOMER_ID
GROUP BY c.CUSTOMER_ID, c.AGE, c.YEARS_LICENSED, c.CLAIMS_HISTORY, c.CREDIT_SCORE
"""

customer_features_df = session.sql(customer_features_sql)
customer_features_df.show(5)

In [ ]:
online_config = OnlineConfig(enable=True, target_lag="60 minutes")

customer_fv = FeatureView(
    name="CUSTOMER_RISK_FEATURES",
    entities=[customer_entity],
    feature_df=customer_features_df,
    timestamp_col="UPDATED_AT",
    refresh_freq="60 minutes",
    refresh_mode="AUTO",
    desc="Customer risk and profile features for insurance pricing with online serving",
    online_config=online_config
)

customer_fv = fs.register_feature_view(
    feature_view=customer_fv,
    version="v1"
)

print("Feature View 'CUSTOMER_RISK_FEATURES' registered with Online Feature Store enabled")
print(f"Online target lag: 1 minute")
print(f"Offline refresh freq: 5 minutes")

In [11]:
print("Registered Feature Views:")
fs.list_feature_views().to_pandas()

## 4. Prepare Training Dataset

In [12]:
spine_df = (
    session.table(f"{DATABASE}.{SCHEMA}.POLICIES")
    .join(
        session.table(f"{DATABASE}.{SCHEMA}.CUSTOMERS").select("CUSTOMER_ID", "GENDER", "STATE"),
        on="CUSTOMER_ID"
    )
    .with_column("CAR_AGE", F.lit(current_year_val) - F.col("CAR_YEAR"))
    .select(
        "CUSTOMER_ID",
        "CAR_MAKE",
        "CAR_MODEL",
        "CAR_AGE",
        "KILOMETERS",
        "ENGINE_SIZE",
        "FUEL_TYPE",
        "TRANSMISSION",
        "COVERAGE_TYPE",
        "ESTIMATED_CAR_VALUE",
        "GENDER",
        "STATE",
        "ANNUAL_PREMIUM",
        F.col("UPDATED_AT").cast("timestamp").alias("TS")
    )
)

print(f"Spine DataFrame created with {spine_df.count()} rows")
print("Spine columns (join key + policy data + label):")
print(spine_df.columns)
spine_df.show(5)

In [13]:
training_dataset = fs.generate_dataset(
    name=f"{DATABASE}.{SCHEMA}.CAR_INSURANCE_TRAINING_DATASET",
    spine_df=spine_df,
    features=[customer_fv],
    spine_timestamp_col="TS",
    spine_label_cols=["ANNUAL_PREMIUM"],
    desc="Training dataset for car insurance premium prediction with customer risk features"
)

print("Snowflake Dataset created successfully!")
print(f"Dataset name: {training_dataset.fully_qualified_name}")

training_dataset_df = training_dataset.read.to_snowpark_dataframe()
print(f"Training dataset rows: {training_dataset_df.count()}")
training_dataset_df.show(5)

In [14]:
training_df = training_dataset_df.to_pandas()
print(f"Training DataFrame shape: {training_df.shape}")
print("\nDataset Statistics:")
training_df.describe()

## 5. Feature Engineering and Model Training

In [15]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import xgboost as xgb

categorical_cols = ['CAR_MAKE', 'CAR_MODEL', 'FUEL_TYPE', 'TRANSMISSION', 'COVERAGE_TYPE', 'GENDER', 'STATE']
numeric_cols = ['CAR_AGE', 'KILOMETERS', 'ENGINE_SIZE', 'ESTIMATED_CAR_VALUE',
                'AGE', 'YEARS_LICENSED', 'CLAIMS_HISTORY', 'CREDIT_SCORE',
                'RISK_SCORE', 'AVG_CLAIMS_PER_YEAR', 'TOTAL_POLICIES', 'AVG_CAR_AGE', 
                'AVG_KILOMETERS', 'TOTAL_CAR_VALUE', 'AVG_DEDUCTIBLE']

label_encoders = {}
training_encoded = training_df.copy()

for col in categorical_cols:
    le = LabelEncoder()
    training_encoded[f"{col}_ENCODED"] = le.fit_transform(training_encoded[col].astype(str))
    label_encoders[col] = le

feature_cols = numeric_cols + [f"{col}_ENCODED" for col in categorical_cols]

X = training_encoded[feature_cols]
y = training_encoded['ANNUAL_PREMIUM']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training set: {X_train.shape}")
print(f"Test set: {X_test.shape}")
print(f"\nFeatures: {feature_cols}")

In [16]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

X_train_scaled = pd.DataFrame(X_train_scaled, columns=feature_cols, index=X_train.index)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=feature_cols, index=X_test.index)

print("StandardScaler applied")
print(f"Scaled feature means (should be ~0): {X_train_scaled.mean().mean():.6f}")
print(f"Scaled feature stds (should be ~1): {X_train_scaled.std().mean():.6f}")

In [17]:
xgb_model = xgb.XGBRegressor(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

xgb_model.fit(
    X_train_scaled, 
    y_train,
    eval_set=[(X_test_scaled, y_test)],
    verbose=20
)

print("\nXGBoost model trained!")

In [18]:
y_pred = xgb_model.predict(X_test_scaled)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("Model Performance Metrics:")
print(f"  RMSE: ${rmse:.2f}")
print(f"  MAE:  ${mae:.2f}")
print(f"  R2:   {r2:.4f}")

metrics = {
    "rmse": float(rmse),
    "mae": float(mae),
    "r2": float(r2)
}

In [19]:
feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': xgb_model.feature_importances_
}).sort_values('importance', ascending=False)

print("Top 10 Most Important Features:")
feature_importance.head(10)

## 6. Create Custom Model for Registry

In [20]:
from snowflake.ml.model import custom_model

class CarInsurancePricingModel(custom_model.CustomModel):
    
    def __init__(self, context: custom_model.ModelContext) -> None:
        super().__init__(context)
        import xgboost as xgb
        import pickle
        
        with open(context.path("xgb_model.ubj"), "rb") as f:
            self.model = pickle.load(f)
        with open(context.path("scaler.pkl"), "rb") as f:
            self.scaler = pickle.load(f)
        with open(context.path("label_encoders.pkl"), "rb") as f:
            self.label_encoders = pickle.load(f)
        with open(context.path("feature_cols.pkl"), "rb") as f:
            self.feature_cols = pickle.load(f)
    
    @custom_model.inference_api
    def transform(self, input_df: pd.DataFrame) -> pd.DataFrame:
        df = input_df.copy()
        
        categorical_cols = ['CAR_MAKE', 'CAR_MODEL', 'FUEL_TYPE', 'TRANSMISSION', 
                           'COVERAGE_TYPE', 'GENDER', 'STATE']
        
        for col in categorical_cols:
            if col in df.columns:
                le = self.label_encoders[col]
                df[f"{col}_ENCODED"] = df[col].apply(
                    lambda x: le.transform([str(x)])[0] if str(x) in le.classes_ else 0
                )
        
        available_cols = [c for c in self.feature_cols if c in df.columns]
        X = df[available_cols]
        X_scaled = self.scaler.transform(X)
        
        return pd.DataFrame(X_scaled, columns=available_cols)
    
    @custom_model.inference_api
    def predict(self, input_df: pd.DataFrame) -> pd.DataFrame:
        import numpy as np
        
        X_scaled = self.transform(input_df)
        
        predictions = self.model.predict(X_scaled.values)
        predictions = np.maximum(predictions, 400)
        predictions = np.minimum(predictions, 5000)
        
        return pd.DataFrame({"PREDICTED_PREMIUM": predictions})

print("Custom model class defined")

In [21]:
import tempfile
import pickle
import os as os_module

model_artifacts_dir = tempfile.mkdtemp()

with open(os_module.path.join(model_artifacts_dir, "xgb_model.ubj"), "wb") as f:
    pickle.dump(xgb_model, f)

with open(os_module.path.join(model_artifacts_dir, "scaler.pkl"), "wb") as f:
    pickle.dump(scaler, f)

with open(os_module.path.join(model_artifacts_dir, "label_encoders.pkl"), "wb") as f:
    pickle.dump(label_encoders, f)

with open(os_module.path.join(model_artifacts_dir, "feature_cols.pkl"), "wb") as f:
    pickle.dump(feature_cols, f)

print(f"Model artifacts saved to: {model_artifacts_dir}")
print(f"Files: {os_module.listdir(model_artifacts_dir)}")

In [24]:
from snowflake.ml.model.model_signature import infer_signature

input_cols = ['CAR_AGE', 'KILOMETERS', 'ENGINE_SIZE', 'ESTIMATED_CAR_VALUE',
              'AGE', 'YEARS_LICENSED', 'CLAIMS_HISTORY', 'CREDIT_SCORE',
              'RISK_SCORE', 'AVG_CLAIMS_PER_YEAR', 'TOTAL_POLICIES', 'AVG_CAR_AGE', 
              'AVG_KILOMETERS', 'TOTAL_CAR_VALUE', 'AVG_DEDUCTIBLE',
              'CAR_MAKE', 'CAR_MODEL', 'FUEL_TYPE', 'TRANSMISSION', 'COVERAGE_TYPE',
              'GENDER', 'STATE']

# Signature for the predict function
signature = infer_signature(
    input_data=training_dataset_df.select(input_cols).limit(100),
    output_data=training_dataset_df.select("ANNUAL_PREMIUM").with_column_renamed("ANNUAL_PREMIUM", "PREDICTED_PREMIUM").limit(100)
)

model_context = custom_model.ModelContext(
    artifacts={
        "xgb_model.ubj": os_module.path.join(model_artifacts_dir, "xgb_model.ubj"),
        "scaler.pkl": os_module.path.join(model_artifacts_dir, "scaler.pkl"),
        "label_encoders.pkl": os_module.path.join(model_artifacts_dir, "label_encoders.pkl"),
        "feature_cols.pkl": os_module.path.join(model_artifacts_dir, "feature_cols.pkl")
    }
)

pricing_model = CarInsurancePricingModel(model_context)

# Signature for the transform function
# Run transform on sample data to get the actual output shape/types
sample_input = training_dataset_df.select(input_cols).limit(100).to_pandas()
transform_output = pricing_model.transform(sample_input)

transform_signature = infer_signature(
    input_data=sample_input,
    output_data=transform_output
)

print("Predict signature:")
print(signature)
print("\nTransform signature:")
print(transform_signature)

## 7. Register Model to Snowflake Model Registry

In [25]:
from snowflake.ml.registry import Registry
from snowflake.ml.model import type_hints

registry = Registry(session=session, database_name=DATABASE, schema_name=SCHEMA)

print(f"Registry opened: {DATABASE}.{SCHEMA}")

In [26]:


test_sample = training_dataset_df.select(input_cols).limit(5).to_pandas()
test_pred = pricing_model.predict(test_sample)
print("Local model test predictions:")
print(test_pred)

In [27]:
mv = registry.log_model(
    pricing_model,
    model_name="CAR_INSURANCE_PRICING_MODEL",
    version_name="v4",
    signatures={"predict": signature, "transform": transform_signature},
    sample_input_data=training_dataset_df.select(input_cols).limit(100),
    conda_dependencies=["xgboost", "scikit-learn", "pandas", "numpy"],
    target_platforms=["WAREHOUSE", "SNOWPARK_CONTAINER_SERVICES"],
    comment="XGBoost model for car insurance premium prediction with StandardScaler preprocessing",
    metrics=metrics,
)

print(f"Model registered: {mv.model_name} version {mv.version_name}")
print(f"Metrics: {mv.show_metrics()}")

In [28]:
print("Registered models:")
registry.show_models()

## 8. Test Model Predictions

In [29]:
def calculate_risk_features_for_new_customer(age, years_licensed, credit_score=700):
    """Calculate risk features for new customers (same logic as Feature View)"""
    claims_history = 0
    risk_score = (
        (30 if age < 25 else (20 if age > 65 else 0)) +
        (claims_history * 15) +
        max(0, 40 - years_licensed) +
        max(0, (700 - credit_score) / 10)
    )
    avg_claims_per_year = 0.0
    credit_tier = ('Excellent' if credit_score >= 750 else 
                   'Good' if credit_score >= 700 else 
                   'Fair' if credit_score >= 650 else 'Poor')
    return {
        'CLAIMS_HISTORY': claims_history,
        'CREDIT_SCORE': credit_score,
        'RISK_SCORE': round(risk_score, 2),
        'AVG_CLAIMS_PER_YEAR': round(avg_claims_per_year, 4),
        'CREDIT_TIER': credit_tier,
        'AVG_DEDUCTIBLE': 500
    }

def prepare_inference_data(customer_id, car_data, fs, session):
    """
    Prepare data for inference:
    - Existing customer: fetch features from Online Feature Store
    - New customer: calculate features with defaults
    """
    if customer_id:
        try:
            customer_features = fs.retrieve_feature_values(
                spine_df=session.create_dataframe([(customer_id,)], schema=["CUSTOMER_ID"]),
                features=[customer_fv]
            ).to_pandas()
            
            if len(customer_features) > 0:
                for col in customer_features.columns:
                    if col != 'CUSTOMER_ID':
                        car_data[col] = customer_features[col].values[0]
                print(f"✓ Found existing customer {customer_id} - using Online Feature Store data")
                return car_data
        except Exception as e:
            print(f"Customer {customer_id} not found in Feature Store: {e}")
    
    print("→ New customer - using default values for risk features")
    risk_features = calculate_risk_features_for_new_customer(
        car_data['AGE'], car_data['YEARS_LICENSED']
    )
    car_data.update(risk_features)
    car_data['TOTAL_POLICIES'] = 0
    car_data['AVG_CAR_AGE'] = car_data['CAR_AGE']
    car_data['AVG_KILOMETERS'] = car_data['KILOMETERS']
    car_data['TOTAL_CAR_VALUE'] = car_data['ESTIMATED_CAR_VALUE']
    return car_data

print("Test Case 1: EXISTING CUSTOMER (using Online Feature Store)")
existing_customer_data = {
    'CUSTOMER_ID': 'CUST_000001',
    'CAR_AGE': 3, 'KILOMETERS': 30000, 'ENGINE_SIZE': 2.0,
    'ESTIMATED_CAR_VALUE': 25000, 'AGE': 35, 'YEARS_LICENSED': 17,
    'CAR_MAKE': 'Toyota', 'CAR_MODEL': 'Camry', 'FUEL_TYPE': 'Gasoline',
    'TRANSMISSION': 'Automatic', 'COVERAGE_TYPE': 'Standard', 'GENDER': 'M', 'STATE': 'CA'
}

print("\nTest Case 2: NEW CUSTOMER (using defaults - no claims, avg credit)")
new_customer_data = {
    'CUSTOMER_ID': None,
    'CAR_AGE': 1, 'KILOMETERS': 5000, 'ENGINE_SIZE': 3.0,
    'ESTIMATED_CAR_VALUE': 55000, 'AGE': 22, 'YEARS_LICENSED': 4,
    'CAR_MAKE': 'BMW', 'CAR_MODEL': '3 Series', 'FUEL_TYPE': 'Gasoline',
    'TRANSMISSION': 'Automatic', 'COVERAGE_TYPE': 'Premium', 'GENDER': 'M', 'STATE': 'NY'
}

test_cases_raw = [existing_customer_data.copy(), new_customer_data.copy()]
test_cases_list = []

for case in test_cases_raw:
    customer_id = case.pop('CUSTOMER_ID')
    prepared = prepare_inference_data(customer_id, case, fs, session)
    test_cases_list.append(prepared)

test_cases = pd.DataFrame(test_cases_list)

for col in categorical_cols:
    le = label_encoders[col]
    test_cases[f"{col}_ENCODED"] = test_cases[col].apply(
        lambda x: le.transform([str(x)])[0] if str(x) in le.classes_ else 0
    )

print("\nPrepared Test Cases (note: CLAIMS_HISTORY, CREDIT_SCORE, AVG_DEDUCTIBLE from Feature Store or defaults):")
test_cases[['CAR_MAKE', 'AGE', 'CLAIMS_HISTORY', 'CREDIT_SCORE', 'RISK_SCORE', 'AVG_DEDUCTIBLE', 'COVERAGE_TYPE']]

In [30]:
predictions = pricing_model.predict(test_cases)

results = pd.DataFrame({
    'Scenario': ['Existing Customer (Feature Store)', 'New Customer (Calculated)'],
    'Description': [
        '35yo, Toyota Camry, 0 claims, Standard',
        '22yo, BMW 3-Series, 2 claims, Premium (High Risk)'
    ],
    'Risk Score': test_cases['RISK_SCORE'].values,
    'Predicted Premium': predictions['PREDICTED_PREMIUM'].values.round(2)
})

print("Predictions:")
results

## 9. Deploy Model as Inference Service (SPCS)

In [31]:
COMPUTE_POOL_NAME = "CC_INFERENCE_CPU_POOL"

session.sql(f"""
CREATE COMPUTE POOL IF NOT EXISTS {COMPUTE_POOL_NAME}
    MIN_NODES = 1
    MAX_NODES = 1
    INSTANCE_FAMILY = CPU_X64_S
    AUTO_RESUME = TRUE
    AUTO_SUSPEND_SECS = 300
""").collect()

print(f"Compute pool {COMPUTE_POOL_NAME} created/verified")

compute_pools = session.sql("SHOW COMPUTE POOLS").collect()
print("\nAvailable Compute Pools:")
for pool in compute_pools:
    print(f"  - {pool['name']}: {pool['instance_family']}, State: {pool['state']}")

In [32]:
SERVICE_NAME = "CAR_INSURANCE_INFERENCE_SVC"

try:
    existing = session.sql(f"SHOW SERVICES LIKE '{SERVICE_NAME}' IN SCHEMA {DATABASE}.{SCHEMA}").collect()
    if existing:
        print(f"Dropping existing service: {SERVICE_NAME}")
        session.sql(f"DROP SERVICE IF EXISTS {DATABASE}.{SCHEMA}.{SERVICE_NAME}").collect()
except:
    pass

print(f"Ready to create service: {SERVICE_NAME}")
print(f"Using compute pool: {COMPUTE_POOL_NAME}")

In [ ]:
!pip list | grep snowflake

In [33]:
mv.create_service(
    service_name=SERVICE_NAME,
    service_compute_pool=COMPUTE_POOL_NAME,
    ingress_enabled=True,
    min_instances = 1,
    max_instances = 1,
    autocapture=True
)

print(f"Service '{SERVICE_NAME}' creation initiated")
print("Note: Service creation takes 5-15 minutes to complete")

In [34]:
import time

print("Checking service status...")
for i in range(10):
    status = session.sql(f"SHOW SERVICES LIKE '{SERVICE_NAME}' IN SCHEMA {DATABASE}.{SCHEMA}").collect()
    if status:
        state = status[0]['status']
        print(f"  Attempt {i+1}: Service state = {state}")
        if state == 'RUNNING':
            print("\nService is RUNNING!")
            break
    else:
        print(f"  Attempt {i+1}: Service not found yet")
    time.sleep(30)
else:
    print("\nService still starting. Check status later with:")
    print(f"  SHOW SERVICES LIKE '{SERVICE_NAME}' IN SCHEMA {DATABASE}.{SCHEMA};")

In [35]:
#session.sql(f"ALTER SERVICE {SERVICE_NAME} SET MIN_INSTANCES = 1").collect()

In [36]:
print("Model Services:")
mv.list_services()

## 10. Create Gateway for Stable URL Access

A Gateway provides a stable URL that routes to SPCS endpoints. Unlike direct service endpoints that change when services are recreated, the Gateway URL remains constant, making it ideal for production integrations.

In [37]:
role_name = session.get_current_role()
print (f'Current role: {role_name}')

session.sql(f'GRANT SERVICE ROLE {DATABASE}.{SCHEMA}.{SERVICE_NAME}!ALL_ENDPOINTS_USAGE TO ROLE {role_name}').collect()

In [38]:
GATEWAY_NAME = "CAR_INSURANCE_GATEWAY"

gateway_spec = f"""
CREATE OR REPLACE GATEWAY {GATEWAY_NAME}
  FROM SPECIFICATION $$
spec:
  type: traffic_split
  split_type: custom
  targets:
  - type: endpoint
    value: {DATABASE}.{SCHEMA}.{SERVICE_NAME}!inference
    weight: 100
$$;
"""

print("Creating Gateway...")
print(f"Gateway Name: {GATEWAY_NAME}")
print(f"Target Service: {DATABASE}.{SCHEMA}.{SERVICE_NAME}")
print(f"\nSQL:\n{gateway_spec}")

session.sql(gateway_spec).collect()
print("\nGateway created successfully!")

In [39]:
print("Verifying Gateway...")
gateway_info = session.sql(f"DESCRIBE GATEWAY {GATEWAY_NAME}").collect()
print("\nGateway Details:")
for row in gateway_info:
    print(f"  {row}")


In [40]:
gateway_info = session.sql(f"DESCRIBE GATEWAY {GATEWAY_NAME}").collect()
if gateway_info:
    gateway_url = f"https://{gateway_info[0]['ingress_url']}"
    print("="*60)
    print("GATEWAY ACCESS INFORMATION")
    print("="*60)
    print(f"\nGateway Name: {GATEWAY_NAME}")
    print(f"Gateway URL:  {gateway_url}")
    print(f"\nTarget Endpoint: {DATABASE}.{SCHEMA}.{SERVICE_NAME}!predict")
    print("\nUsage Example (Python):")
    print(f'''
import requests
import json

gateway_url = "{gateway_url}"
headers = {{
    "Authorization": "Snowflake Token=\\"<YOUR_JWT_TOKEN>\\"",
    "Content-Type": "application/json"
}}

data = {{
    "data": [[
        35,           # age
        "Male",       # gender  
        2.0,          # years_since_claim
        "Sedan",      # vehicle_type
        12000,        # annual_mileage
        750,          # credit_score
        5,            # years_as_customer
        10,           # vehicle_age
        "Comprehensive", # coverage_type
        "Urban",      # location
        1             # has_previous_claim
    ]]
}}

response = requests.post(gateway_url, headers=headers, json=data)
prediction = response.json()
print(f"Predicted Premium: ${{prediction['data'][0][0]:.2f}}")
''')
else:
    print("Gateway not found. Please check the gateway was created successfully.")

## 12. Batch Inference via SPCS Service (Online Inference)

Call the model's `predict` function routed through the SPCS inference service running on the compute pool.
This ensures predictions execute on the container service (compute pool) rather than a virtual warehouse.

**Key**: passing `service_name` to `mv.run()` routes inference to the SPCS service.

In [41]:
# Resume the service if it's suspended
session.sql(f"""
    ALTER SERVICE {DATABASE}.{SCHEMA}.{SERVICE_NAME} RESUME
""").collect()

print(f"Service {SERVICE_NAME} resume requested")

In [42]:
# Wait for the service to be READY before calling predict
import time

for i in range(30):
    status = session.sql(f"""
        SELECT SYSTEM$GET_SERVICE_STATUS('{DATABASE}.{SCHEMA}.{SERVICE_NAME}')
    """).collect()[0][0]
    print(f"  [{i+1}] Service status: {status}")
    if "READY" in status.upper():
        print("Service is READY!")
        break
    time.sleep(10)
else:
    print("Warning: Service did not reach READY state within timeout")

In [43]:
# Batch inference routed through the SPCS service (compute pool)
# Get the model version from the registry
mv = registry.get_model("CAR_INSURANCE_PRICING_MODEL").version("V4")

# Prepare a batch input DataFrame with the columns matching the model signature
input_cols = [
    'CAR_AGE', 'KILOMETERS', 'ENGINE_SIZE', 'ESTIMATED_CAR_VALUE',
    'AGE', 'YEARS_LICENSED', 'CLAIMS_HISTORY', 'CREDIT_SCORE',
    'RISK_SCORE', 'AVG_CLAIMS_PER_YEAR', 'TOTAL_POLICIES', 'AVG_CAR_AGE',
    'AVG_KILOMETERS', 'TOTAL_CAR_VALUE', 'AVG_DEDUCTIBLE',
    'CAR_MAKE', 'CAR_MODEL', 'FUEL_TYPE', 'TRANSMISSION', 'COVERAGE_TYPE',
    'GENDER', 'STATE'
]

batch_input_df = training_dataset_df.select(input_cols).limit(20)

# Run predictions on the SPCS service by specifying service_name
# Without service_name, this would run on the warehouse instead
predictions = mv.run(
    batch_input_df,
    function_name="predict",
    service_name=SERVICE_NAME
)

print(f"Batch predictions via SPCS service ({SERVICE_NAME}):")
predictions.show()

## 11. Summary

### Created Resources:
- **Database/Schema**: `CC_ML_INSURANCE.CAR_PRICING`
- **Tables**: `CUSTOMERS` (5,000 rows), `POLICIES` (8,000 rows)
- **Feature Store**: `CUSTOMER_RISK_FEATURES` with Online Feature Store enabled
- **Model**: `CAR_INSURANCE_PRICING_MODEL` (XGBoost with StandardScaler)
- **Service**: `CAR_INSURANCE_INFERENCE_SVC` (SPCS real-time inference)
- **Gateway**: `CAR_INSURANCE_GATEWAY` (Stable URL for API access)

### Benefits of Gateway:
- **Stable URL**: Gateway URL never changes, even if service is recreated
- **Traffic Splitting**: Can route to multiple endpoints with weighted distribution
- **Failover**: Automatic failover if primary endpoint becomes unhealthy

### Next Steps:
1. Wait for service to be READY
2. Use Gateway URL for production API integrations
3. Deploy Streamlit app to test the full pipeline

In [ ]:
print("="*60)
print("CAR INSURANCE ML PIPELINE - SUMMARY")
print("="*60)
print(f"\nDatabase: {DATABASE}")
print(f"Schema: {SCHEMA}")
print(f"\nTables:")
print(f"  - CUSTOMERS: {session.table('CUSTOMERS').count()} rows")
print(f"  - POLICIES: {session.table('POLICIES').count()} rows")
print(f"\nFeature Store:")
print(f"  - Entity: CUSTOMER")
print(f"  - Feature View: CUSTOMER_RISK_FEATURES (Online enabled)")
print(f"\nModel:")
print(f"  - Name: CAR_INSURANCE_PRICING_MODEL")
print(f"  - Version: v1")
print(f"  - Type: XGBoost Regressor")
print(f"  - Preprocessing: StandardScaler + LabelEncoder")
print(f"  - RMSE: ${metrics['rmse']:.2f}")
print(f"  - R2: {metrics['r2']:.4f}")
print(f"\nInference Service:")
print(f"  - Name: {SERVICE_NAME}")
print(f"  - Compute Pool: {COMPUTE_POOL_NAME}")
print(f"\nGateway (Stable API Access):")
print(f"  - Name: {GATEWAY_NAME}")
gateway_result = session.sql(f"SHOW GATEWAYS LIKE '{GATEWAY_NAME}' IN SCHEMA {DATABASE}.{SCHEMA}").collect()
if gateway_result:
    print(f"  - Gateway: {gateway_result}")
print("="*60)